In [31]:
import numpy as np # array-processing package
import pandas as pd # data analysis toolkit
import matplotlib.pyplot as plt # static, animated and interactive visualizations
import seaborn as sns # data visualization library
import re # regular expressions
from sklearn.metrics.pairwise import cosine_similarity # computes cosine similarity between samples
from sklearn.preprocessing import StandardScaler # standardizes features by removing the mean and scaling to unit variance
from scipy.sparse import csr_matrix # sparse matrix package for numeric data
from scipy.stats import pearsonr # computes the pearson correlation coefficient
from scipy.sparse.linalg import svds # singular value decomposition for matrix factorization
from sklearn.metrics import mean_squared_error, mean_absolute_error, precision_score, recall_score, f1_score # evaluation metrics
from sklearn.model_selection import train_test_split # splits arrays or matrices into random train and test subsets
from funcoesaux import fallback_recomendacoes, rerank_diversidade_novidade, get_titulo, genero_preferido, criar_users_ratings

# Set random seed for reproducibility
np.random.seed(2025)

In [3]:
# Note: the datasets were saved as CSV UTF-8 files; sep or delimiter

imdb_movies = pd.read_csv('../datasets/imdb_movies.csv', encoding = 'UTF-8', sep = ';')
imdb_ratings = pd.read_csv('../datasets/imdb_ratings.csv', encoding = 'UTF-8', sep = ';')
imdb_merged_filtrado = pd.read_csv('../datasets/imdb_merged_filtrado.csv', encoding = 'UTF-8', sep = ',')

# Criar users com base nos ratings acima (comentar este código, se já existir o csv com user e não queremos um novo)
#criar_users_ratings(dataset= imdb_merged_filtrado)

In [4]:
#----- User-Based Collaborative Filtering ----------------------------------------------------------------------------------------------------------------------------------------------------

user_ratings = pd.read_csv("user_ratings.csv") 

#user_movie_matrix = user_ratings.pivot(index = ' User ID ', columns = ' Movie ID ', values = ' User Rating ')
#user_movie_matrix = user_movie_matrix.fillna(0)
#print(user_movie_matrix)

train_data = {}
test_data = {}
sem_ratings = []
test_size = 0.6


#Separar os dados de cada utilizador em treino e teste
for user_id in user_ratings[' User ID '].unique():
    filmes_avaliados = user_ratings[user_ratings[' User ID '] == user_id]

    if len(filmes_avaliados) > 5:
        train, test = train_test_split(filmes_avaliados, test_size=test_size, random_state=4465)
        train_data[user_id] = train
        test_data[user_id] = test
    else:
        sem_ratings.append(user_id)

#Criar nova matriz utilizador-filme apenas com os dados de treino
train_ratings = pd.concat(train_data.values())  # Reunir todas as avaliações de treino
train_movie_matrix = train_ratings.pivot(index=' User ID ', columns=' Movie ID ', values=' User Rating ').fillna(0)

#Calcular similaridade entre filmes
user_similarity = cosine_similarity(train_movie_matrix)
#user_similarity = train_movie_matrix.T.corr(method='pearson')
user_similarity_df = pd.DataFrame(user_similarity, index=train_movie_matrix.index, columns=train_movie_matrix.index)
#print(user_similarity_df.head())


def recomendar_UBC(user_id, num_rec, dataset, ratings_df):

    sims = user_similarity_df[user_id].drop(user_id).sort_values(ascending=False)
    top_k = sims[:20]

    #Score inicial
    weighted_ratings = (train_movie_matrix.loc[top_k.index].T.dot(top_k) / top_k.sum())

    #Filtra filmes já vistos
    user_rated = train_movie_matrix.loc[user_id]
    preds = weighted_ratings[user_rated == 0]

    #Obter géneros preferidos
    generos_top = genero_preferido(user_id, ratings_df, dataset)

    #Obter géneros dos filmes recomendados
    filmes_generos = dataset.set_index('imdb_title_id').loc[preds.index][['genre']]
    filmes_generos['genre'] = filmes_generos['genre'].fillna('').str.split(',')
    filmes_generos = filmes_generos['genre'].apply(lambda genres: [g.strip() for g in genres])

    #Aplicar peso dos géneros diretamente ao score
    genre_weights = []
    for generos in filmes_generos:
        if any(g in generos_top for g in generos):
            genre_weights.append(1.5)
        else:
            genre_weights.append(0.7)
    
    #Combina os scores dos vizinhos com os pesos dos géneros
    preds = preds * genre_weights
    preds = preds.sort_values(ascending=False)

    #Remove a metade inferior
    half = int(len(preds) * 0.5)
    if half > 0:
        preds = preds.iloc[:-half]

    #sampled = preds.sample(100, weights=preds, random_state=42)
    #sampled_ids = sampled.index.tolist()
    sampled_ids = preds.index[:50].tolist()

    reranked = rerank_diversidade_novidade(sampled_ids, dataset)

    return reranked[:num_rec]
    #return preds.index[:num_rec].tolist()

In [22]:
# ----- Item-Based Collaborative Filtering ------------------------------------------------------------------------------------------------------------------

# Item Based Collaborative Filtering - recommends items based on similarity with the items that the target user rated.
# The similarity can be computed with Pearson Correlation or Cosine Similarity.



# Choose similarity method
sim_method = 'cosine' # or 'pearson'


# Load ratings
ratings_df = pd.read_csv("user_ratings.csv")


# User-item matrix
user_movie_matrix = ratings_df.pivot(index = ' User ID ', columns = ' Movie ID ', values = ' User Rating ')

popular_movies = imdb_merged_filtrado[imdb_merged_filtrado['total_votes'] > 1000] # added to improve the predictions
valid_movie_ids = [movie_id for movie_id in popular_movies['imdb_title_id'] if movie_id in user_movie_matrix.columns]
user_movie_matrix = user_movie_matrix[valid_movie_ids]

user_movie_matrix = user_movie_matrix.fillna(0)


# Select relevant columns
movies = imdb_merged_filtrado[['imdb_title_id', 'original_title', 'avg_vote', 'total_votes', 'genre']]


# Item-item similarity matrix
if sim_method == 'cosine':
    item_similarity = cosine_similarity(user_movie_matrix.T)
    item_similarity_df = pd.DataFrame(item_similarity, index = user_movie_matrix.columns, columns = user_movie_matrix.columns)

elif sim_method == 'pearson':
    item_similarity_df = user_movie_matrix.corr(method = 'pearson')

else:
    raise ValueError("Invalid similarity method. Use 'cosine' or 'pearson'.")


# Get movie info
def get_movie_info(movie_id):
    row = imdb_merged_filtrado[imdb_merged_filtrado['imdb_title_id'] == movie_id]
    if not row.empty:
        return row[['original_title', 'genre', 'description', 'avg_vote']].values[0]
    else:
        return ["Title not found", "Unknown description", 0]


# Recommend similar movies
def recommend_item(movie_id, num_rec = 10):
    if movie_id not in item_similarity_df.index:
        return None

    rated_by_user = set(ratings_df[ratings_df[' User ID '] == user_id][' Movie ID '])
    similares = item_similarity_df[movie_id].drop(rated_by_user, errors = 'ignore').sort_values(ascending = False).head(num_rec).index

    recommendation = []

    for rec_id in similares:
        title, genre, description, avg_vote = get_movie_info(rec_id)
        user_rating = ratings_df[(ratings_df[' User ID '] == user_id) & (ratings_df[' Movie ID '] == rec_id)][' User Rating '].values
        user_rating = user_rating[0] if len(user_rating) > 0 else None
        recommendation.append({' Movie ID ': rec_id, 'original_title': title, 'avg_vote': avg_vote, 'genre': genre, 'description': description, 'user_rating': user_rating})

    recommendation_df = pd.DataFrame(recommendation)

    # Sort by user_rating (if exists), then by avg_vote
    recommendation_df = recommendation_df.sort_values(by = 'user_rating', ascending = False, na_position = 'last')

    return recommendation_df



# Tryout
user_id = ratings_df[' User ID '].sample(1, random_state = 3112).values[0]
rated_movies_ids = ratings_df[ratings_df[' User ID '] == user_id][' Movie ID '].tolist()

# Choose a random rated movie for testing
np.random.seed(3112)
movie_test = np.random.choice(rated_movies_ids)
title, genre, description, avg_vote = get_movie_info(movie_test)
user_rating = ratings_df[(ratings_df[' User ID '] == user_id) & (ratings_df[' Movie ID '] == movie_test)][' User Rating '].values[0]

print(f"Movie rated by {user_id} selected:\n")
print(f"Title: {title}")
print(f"Genre: {genre}")
print(f"Description: {description}")
print(f"User rating: {user_rating}")
print("\n" + "-"*40 + "\n")

print(f"Recommended movies based on '{title}' (method: {sim_method}):\n")
recs_df = recommend_item(movie_test, num_rec=10)

if recs_df is not None:
    print(recs_df[['original_title', 'genre', 'description']])
else:
    print(f"Unable to generate recommendations for the movie '{title}' (ID: {movie_test}).")

Movie rated by 84 selected:

Title: The Last Man
Genre: Action, Drama, Sci-Fi
Description: Kurt, combat veteran with PTSD and hallucinations, fortifies his home and builds a secret underground shelter due to doomsday like weather changes. He gets a security job to pay for it and his boss' cute daughter for company.
User rating: 6.2

----------------------------------------

Recommended movies based on 'The Last Man' (method: cosine):

                      original_title                       genre  \
0              Ocho apellidos vascos             Comedy, Romance   
1                          I Origins      Drama, Romance, Sci-Fi   
2          Robin Hood: Men in Tights  Adventure, Comedy, Musical   
3                  Strictly Ballroom        Comedy, Drama, Music   
4  Solan og Ludvig - Jul i FlÃ¥klypa           Animation, Family   
5                        Torrid Zone   Action, Adventure, Comedy   
6            Faustrecht der Freiheit              Drama, Romance   
7              Ã

In [23]:
#----- Model-Based Collaborative - Matrix Factorization ------------------------------------------------------------------------------------------------------------------------------

# Matrix Factorization - decomposes the user-item matrix into two lower-dimensional matrices, one for users and one for items.


# Ensure consistency between user-movie matrix and movie metadata
valid_ids = imdb_merged_filtrado['imdb_title_id'].unique()
filtered_matrix = user_movie_matrix.loc[:, user_movie_matrix.columns.isin(valid_ids)]


# Ensure movie metadata only contains relevant movies
relevant_movies = imdb_merged_filtrado[imdb_merged_filtrado['imdb_title_id'].isin(filtered_matrix.columns)].copy()
relevant_movies.drop_duplicates(subset = 'imdb_title_id', inplace = True)
relevant_movies.set_index('imdb_title_id', inplace = True)


# Note: k controls the dimensionality of the latent space and the level of compression
k = min(50, min(filtered_matrix.shape)-1) # number of latent factors


# SVD
# Note: U is the user matrix, S is the singular values (importance of each factor), Vt is the item matrix
def perform_svd(user_movie_matrix, k):
    U, S, Vt = svds(user_movie_matrix, k = k)
    S = np.diag(S) # convert singular values into a diagonal matrix to allow matrix multiplication
    return U, S, Vt

U, S, Vt = perform_svd(filtered_matrix.values, k)


# Predicted ratings matrix
predicted_ratings = np.dot(np.dot(U, S), Vt)
predicted_ratings_df = pd.DataFrame(predicted_ratings, index = filtered_matrix.index, columns = filtered_matrix.columns)


def recommend_movies_svd(user_id, n = 10):
    if user_id not in predicted_ratings_df.index:
        return f"User {user_id} not found."

    # Get the movies with highest predicted ratings for the user
    user_ratings = predicted_ratings_df.loc[user_id].sort_values(ascending = False)

    # Filter out movies already rated by the user
    already_rated = filtered_matrix.loc[user_id][filtered_matrix.loc[user_id] > 0].index
    recommended_movies = user_ratings.drop(index = already_rated, errors = 'ignore').head(n)

    # Filter only valid movies
    valid_recs = recommended_movies.index.intersection(relevant_movies.index)

    recommended_df = relevant_movies.loc[valid_recs][['original_title', 'genre', 'description']]

    return recommended_df


# Tryout
user_example = filtered_matrix.sample(n = 1, random_state = 3112).index[0] # random user


# Movies previously rated by the user
rated_ids = filtered_matrix.loc[user_example][filtered_matrix.loc[user_example] > 0].index
rated_ids = rated_ids.intersection(relevant_movies.index)

user_movies = relevant_movies.loc[rated_ids][['original_title', 'genre', 'description']].copy()
user_movies['user_rating'] = filtered_matrix.loc[user_example, rated_ids].values
user_movies = user_movies.sort_values(by = 'user_rating', ascending = False)

print(f"Recommendation for: {user_example}\n")
print("Movies previously rated by the user:")
print(user_movies)

print("\n--------------------\n")
print("Recommended movies:")
print(recommend_movies_svd(user_example, n = 10))

Recommendation for: 80

Movies previously rated by the user:
                    original_title                        genre  \
tt0495040  Kunsten at grÃ¦de i kor                Comedy, Drama   
tt0756683       The Man from Earth       Drama, Fantasy, Sci-Fi   
tt0424986                 El Nazer                       Comedy   
tt0037793         House of Dracula      Fantasy, Horror, Sci-Fi   
tt0038166     The Three Caballeros    Animation, Comedy, Family   
...                            ...                          ...   
tt5617916            Airplane Mode                       Comedy   
tt7388562  Paul, Apostle of Christ  Adventure, Biography, Drama   
tt6417204       Shatamanam Bhavati                        Drama   
tt0015214           Paris qui dort                       Sci-Fi   
tt7203412              Wing Mirror                Comedy, Drama   

                                                 description  user_rating  
tt0495040  Southern Jutland, DK, early 1970s: A dysfuncti.

In [24]:
####################################################################################################################################################################################
# Cold Start Problem

# Note: The cold start problem occurs when a recommender system cannot make accurate recommendations due to the lack of data about new users or new items.


# ----- New Users --------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Cosine similarity between users
cos_sim = cosine_similarity(user_movie_matrix)


# Recommend the most popular items
def recommend_popular_items(user_movie_matrix, n_recommendations = 5):
    item_popularity = user_movie_matrix.astype(bool).sum(axis = 0) # count how many users rated each item
    popular_items = item_popularity.sort_values(ascending = False).head(n_recommendations).index.tolist() # top n most popular items
    return popular_items


# Recommend popular items for a new user
def recommend_for_new_user(user_movie_matrix, n_recommendations = 5):
    return recommend_popular_items(user_movie_matrix, n_recommendations)


# Recommend similar items for an existing user based on the items they have rated
# Note: if user has no ratings or does not exist, return empty list. This way we can check if the picked user already existed.
def recommend_similar_items_based_on_popularity(user_id, user_movie_matrix, n_recommendations = 5):
    if user_id not in user_movie_matrix.index:
        return [] # cold start detected

    user_ratings = user_movie_matrix.loc[user_id]
    rated_items = user_ratings[user_ratings > 0].index

    if len(rated_items) == 0:
        return [] # user has no rated items, can't compute similarity

    candidate_items = user_ratings[user_ratings == 0].index # items the user has not rated

    similar_items = []

    # For each item rated by the user, compute its similarity to unrated items
    for item in rated_items:
        item_vector = user_movie_matrix[item].values.reshape(1, -1) # rated item
        candidate_vectors = user_movie_matrix[candidate_items].values.T # candidate items
        similarities = cosine_similarity(item_vector, candidate_vectors).flatten() # cosine similarity

        similar_items.extend(zip(candidate_items, similarities)) # store the candidate items and their similarity scores

    similar_items = sorted(similar_items, key = lambda x: x[1], reverse = True)

    # Top n unique recommended items
    recommended = []
    seen = set()

    # Iterate over sorted similar items and add unique ones to the recommendation list
    for item, _ in similar_items:
        if item not in seen:
            recommended.append(item)
            seen.add(item)
        if len(recommended) == n_recommendations:
            break

    return recommended


# Tryout
new_user_id = 500 # this user does not exist (cold start scenario)
popular_recommendations = recommend_for_new_user(user_movie_matrix)
similar_item_recommendations = recommend_similar_items_based_on_popularity(new_user_id, user_movie_matrix)


# Convert movies IDs to titles
popular_titles = imdb_merged_filtrado[imdb_merged_filtrado['imdb_title_id'].isin(popular_recommendations)]['original_title'].tolist()
similar_titles = imdb_merged_filtrado[imdb_merged_filtrado['imdb_title_id'].isin(similar_item_recommendations)]['original_title'].tolist()


print(f"Popular recommendations for new user {new_user_id}: {popular_titles}")
print(f"Similarity-based recommendations for new user {new_user_id}: {similar_titles}")

Popular recommendations for new user 500: ['Juve contre FantÃ´mas', 'Nosferatu, eine Symphonie des Grauens', 'Mahanadhi', 'The Man from Earth', 'Bright']
Similarity-based recommendations for new user 500: []


In [25]:
# ----- New Movies --------------------------------------------------------------------------------------------------------------------------------------------------------------------


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd


# Combine textual features into a single string
def combine_features(row):
    return ' '.join([str(row['genre']), str(row['description']), str(row['director']), str(row['actors']), str(row['writer'])])


# Fill missing values with empty strings
imdb_merged_filtrado[['genre', 'description', 'director', 'actors', 'writer']] = imdb_merged_filtrado[['genre', 'description', 'director', 'actors', 'writer']].fillna('')


# Create a new column for the textual features
imdb_merged_filtrado['combined_features'] = imdb_merged_filtrado.apply(combine_features, axis = 1)


# TF-IDF (term frequency-inverse document frequency)
# Note: F-IDF measures the importance of string representations such as words, phrases and more
# Note: we remove common stop words (e.g., the, and, or of) to reduce noise
tfidf = TfidfVectorizer(stop_words = 'english') # converts a collection of raw documents to a matrix of TF-IDF features
tfidf_matrix = tfidf.fit_transform(imdb_merged_filtrado['combined_features']) # (n_movies, n_features)


# Recommend similar movies to a new movie
def recommend_similar_movies_for_new_item(new_movie_data, n_recommendations = 10):
    # Note: new_movie_data is a dict with the same keys used in combine_features
    new_movie_text = ' '.join([str(new_movie_data.get('genre', '')), str(new_movie_data.get('description', '')), str(new_movie_data.get('director', '')), str(new_movie_data.get('actors', '')), str(new_movie_data.get('writer', ''))])

    # Transform new movie text
    new_tfidf = tfidf.transform([new_movie_text]) # transforms text into a sparse matrix of n-gram counts

    # Cosine similarity between new movie and all existing ones
    similarity_scores = cosine_similarity(new_tfidf, tfidf_matrix).flatten()

    # Get indices of top N most similar movies
    top_indices = similarity_scores.argsort()[::-1][:n_recommendations]

    return imdb_merged_filtrado.iloc[top_indices][['imdb_title_id', 'title', 'genre', 'description']]


# Tryout 1
new_movie_info = {'genre': 'Action Crime', 'description': 'A secret agent embarks on a mission to stop an international crime syndicate.', 'director': 'Rob Cohen', 'actors': 'Tom Cruise, Vin Diesel', 'writer': 'Bruce Geller'}

recommendations = recommend_similar_movies_for_new_item(new_movie_info)

print("Recommendations for the new movie:")
print(recommendations)


# Tryout 2
new_movie_info_try = {'genre': 'Comedy', 'description': 'A summer to remember, full of music and joy.', 'director': 'Phyllida Lloyd', 'actors': 'Emmy Rossum', 'writer': 'Catherine Johnson'}

recommendations = recommend_similar_movies_for_new_item(new_movie_info_try)

print('\n ----------------------------------')
print("Recommendations for the new movie:")
print(recommendations)

Recommendations for the new movie:
      imdb_title_id                          title  \
33108     tt0149171                         Strays   
40863     tt0295701                            xXx   
37418     tt0232500               Fast and Furious   
76126     tt4912910  Mission: Impossible - Fallout   
31291     tt0120755         Mission: Impossible II   
29967     tt0117060            Mission: Impossible   
28949     tt0113189                      GoldenEye   
31331     tt0120815        Salvate il soldato Ryan   
59809     tt1596343               Fast & Furious 5   
83401     tt7903530                   We Die Young   

                             genre  \
33108                 Crime, Drama   
40863  Action, Adventure, Thriller   
37418      Action, Crime, Thriller   
76126  Action, Adventure, Thriller   
31291  Action, Adventure, Thriller   
29967  Action, Adventure, Thriller   
28949  Action, Adventure, Thriller   
31331                   Drama, War   
59809     Action, Adventure,

In [26]:
##########################################################################################################################################################################
# Parameter Tuning and Evaluation

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds


# Parameters
param_grid_user_based = {'similarity': ['cosine', 'pearson'], 'n_neighbors': [5, 10, 15]}
param_grid_item_based = {'similarity': ['cosine', 'pearson'], 'n_neighbors': [5, 10, 15]}
param_grid_matrix_factorization = {'k': [30, 40, 50, 60]}


def evaluate_predictions(actual_matrix, predicted_matrix):
    # Ensure both matrices are float
    actual_matrix = actual_matrix.astype(float)
    predicted_matrix = predicted_matrix.astype(float)

    # Create mask where actual ratings exist (non-NaN)
    mask = actual_matrix.notna()

    # Actual and predicted values
    actual = actual_matrix[mask].values.flatten()
    predicted = predicted_matrix[mask].values.flatten()

    # RMSE and MAE
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    return round(rmse, 4), round(mae, 4)


# User-Based Collaborative Filtering
def evaluate_user_based_model(user_movie_matrix, similarity_metric = 'cosine', n_neighbors = 5):
    matrix_filled = user_movie_matrix.fillna(0).astype(float)

    if similarity_metric == 'cosine':
        sim_matrix = cosine_similarity(matrix_filled)
    elif similarity_metric == 'pearson':
        sim_matrix = matrix_filled.T.corr(method = 'pearson').fillna(0).values

    sim_df = pd.DataFrame(sim_matrix, index = user_movie_matrix.index, columns = user_movie_matrix.index)

    # Top-k neighbors
    top_k_similarities = pd.DataFrame(0.0, index = sim_df.index, columns = sim_df.columns)
    for i in sim_df.index:
        top_k = sim_df.loc[i].nlargest(n_neighbors + 1).drop(i, errors = 'ignore') # +1 includes self, errors = 'ignore': invalid parsing will return the input
        top_k_similarities.loc[i, top_k.index] = top_k.values

    # Normalize weights
    weights = top_k_similarities.div(top_k_similarities.sum(axis = 1), axis = 0).fillna(0)

    # Predict ratings
    predicted_ratings = weights.dot(matrix_filled)

    return pd.DataFrame(predicted_ratings, index = user_movie_matrix.index, columns = user_movie_matrix.columns)


# Item-Based Collaborative Filtering
def evaluate_item_based_model(user_movie_matrix, similarity_metric = 'cosine', n_neighbors = 5):
    matrix_filled = user_movie_matrix.fillna(0).astype(float)

    if similarity_metric == 'cosine':
        sim_matrix = cosine_similarity(matrix_filled.T)
    elif similarity_metric == 'pearson':
        sim_matrix = matrix_filled.corr(method='pearson').fillna(0).values

    sim_df = pd.DataFrame(sim_matrix, index = user_movie_matrix.columns, columns = user_movie_matrix.columns)

    # Top-k neighbors
    top_k_weights = pd.DataFrame(0.0, index = sim_df.index, columns = sim_df.columns)
    for i in sim_df.index:
        top_k = sim_df.loc[i].nlargest(n_neighbors + 1).drop(i, errors = 'ignore')
        top_k_weights.loc[i, top_k.index] = top_k.values

    # Normalize weights
    top_k_weights = top_k_weights.div(top_k_weights.sum(axis = 1), axis = 0).fillna(0)

    # Predict ratings
    predicted_ratings = top_k_weights.dot(matrix_filled.T).T

    return pd.DataFrame(predicted_ratings, index = user_movie_matrix.index, columns = user_movie_matrix.columns)


# Matrix Factorization (SVD)
def evaluate_matrix_factorization(user_movie_matrix, k = 20):
    matrix_filled = user_movie_matrix.fillna(0).astype(float)

    U, S, Vt = svds(matrix_filled.values, k = k)
    S_diag = np.diag(S)
    predicted_ratings = np.dot(np.dot(U, S_diag), Vt)

    predicted_df = pd.DataFrame(predicted_ratings, index = user_movie_matrix.index, columns = user_movie_matrix.columns)

    return predicted_df.fillna(0)


# Evaluation Execution
print("Manual Tuning - User-Based Collaborative Filtering")
for sim in param_grid_user_based['similarity']:

    for k in param_grid_user_based['n_neighbors']:
        predicted = evaluate_user_based_model(user_movie_matrix, similarity_metric = sim, n_neighbors = k)
        rmse, mae = evaluate_predictions(user_movie_matrix, predicted)

        print(f"Similarity: {sim}, Neighbors: {k} --> RMSE: {rmse}, MAE: {mae}")


print("\nManual Tuning - Item-Based Collaborative Filtering")
for sim in param_grid_item_based['similarity']:

    for k in param_grid_item_based['n_neighbors']:
        predicted = evaluate_item_based_model(user_movie_matrix, similarity_metric = sim, n_neighbors = k)
        rmse, mae = evaluate_predictions(user_movie_matrix, predicted)

        print(f"Similarity: {sim}, Neighbors: {k} --> RMSE: {rmse}, MAE: {mae}")


print("\nManual Tuning - Matrix Factorization (SVD)")
for k in param_grid_matrix_factorization['k']:
    predicted = evaluate_matrix_factorization(user_movie_matrix, k = k)
    rmse, mae = evaluate_predictions(user_movie_matrix, predicted)

    print(f"Latent factors (k): {k} --> RMSE: {rmse}, MAE: {mae}")


Manual Tuning - User-Based Collaborative Filtering
Similarity: cosine, Neighbors: 5 --> RMSE: 1.0641, MAE: 0.2637
Similarity: cosine, Neighbors: 10 --> RMSE: 1.022, MAE: 0.2631
Similarity: cosine, Neighbors: 15 --> RMSE: 1.0077, MAE: 0.2631
Similarity: pearson, Neighbors: 5 --> RMSE: 1.064, MAE: 0.2636
Similarity: pearson, Neighbors: 10 --> RMSE: 1.022, MAE: 0.2629
Similarity: pearson, Neighbors: 15 --> RMSE: 1.0077, MAE: 0.2629

Manual Tuning - Item-Based Collaborative Filtering
Similarity: cosine, Neighbors: 5 --> RMSE: 0.5968, MAE: 0.1002
Similarity: cosine, Neighbors: 10 --> RMSE: 0.6086, MAE: 0.1022
Similarity: cosine, Neighbors: 15 --> RMSE: 0.6249, MAE: 0.1039
Similarity: pearson, Neighbors: 5 --> RMSE: 0.7152, MAE: 0.1209
Similarity: pearson, Neighbors: 10 --> RMSE: 0.7787, MAE: 0.1339
Similarity: pearson, Neighbors: 15 --> RMSE: 0.8292, MAE: 0.143

Manual Tuning - Matrix Factorization (SVD)
Latent factors (k): 30 --> RMSE: 0.8028, MAE: 0.3885
Latent factors (k): 40 --> RMSE: 0

In [40]:
# Test for Hybrid User-Based Collaborative Filtering (Already with tuning and its evaluation metric)

def avaliar_recomendacoes_com_genero(user_id, recomendacoes, ground_truth, dataset, genero_peso=0.5, filme_peso=0.5):
    filmes_certos = sum(1 for filme in recomendacoes if filme in ground_truth)
    
    generos_user = genero_preferido(user_id, ratings_df, dataset)

    #Pega os géneros dos filmes recomendados
    filmes_rec_genero = dataset[dataset['imdb_title_id'].isin(recomendacoes)][['imdb_title_id', 'genre']]
    filmes_rec_genero['genre'] = filmes_rec_genero['genre'].str.split(',')
    filmes_rec_genero = filmes_rec_genero.explode('genre')
    filmes_rec_genero['genre'] = filmes_rec_genero['genre'].str.strip()

    #Ver quantos dos filmes recomendados contêm o género preferido
    genero_certo = filmes_rec_genero['genre'].isin(generos_user).sum()

    total_recs = len(recomendacoes)
    filme_score = filmes_certos / total_recs
    genero_score = genero_certo / total_recs

    score_final = (filme_peso * filme_score) + (genero_peso * genero_score)
    return score_final


#Teste para todos os users no dataset que tem ratings suficientes
scores_array = []
for user_id in test_data.keys():
    ground_truth = test_data[user_id][' Movie ID '].tolist()  
    recomendacoes = recomendar_UBC(user_id, 10, imdb_merged_filtrado, ratings_df)  
    score = avaliar_recomendacoes_com_genero(user_id, recomendacoes, ground_truth, imdb_merged_filtrado)
    if score < 0.1:
        recomendacoes = fallback_recomendacoes(user_id, imdb_merged_filtrado, ratings_df, train_movie_matrix, popular_movies)
        score = avaliar_recomendacoes_com_genero(user_id, recomendacoes, ground_truth, imdb_merged_filtrado)
    scores_array.append(score)
#print([round(float(score), 3) for score in scores_array])
print(f"Avaliação Média: {np.average(scores_array):.2f}")



Avaliação Média: 0.46


In [39]:
def mostrar_recomendacoes_user(user_id, dataset, ratings_df, test_data, train_movie_matrix, popular_movies, n_recs=10):
    print(f"\n--- Recomendação para o Utilizador {user_id} ---")

    # Obter ground truth
    ground_truth = test_data[user_id][' Movie ID '].tolist()

    # Gerar recomendações
    recomendacoes = recomendar_UBC(user_id, n_recs, dataset, ratings_df)

    # Avaliar score inicial
    score = avaliar_recomendacoes_com_genero(user_id, recomendacoes, ground_truth, dataset)

    # Se score for fraco, usar fallback
    if score < 0.1:
        print("(Recomendação principal fraca. A usar fallback.)")
        recomendacoes = fallback_recomendacoes(user_id, dataset, ratings_df, train_movie_matrix, popular_movies)
        score = avaliar_recomendacoes_com_genero(user_id, recomendacoes, ground_truth, dataset)

    # Mostrar os títulos dos filmes recomendados
    print("Filmes Recomendados:")
    for movie_id in recomendacoes:
        try:
            titulo = get_titulo(movie_id, dataset)
            print(f" - {titulo} ({movie_id})")
        except:
            print(f" - [Título não encontrado] ({movie_id})")

    print(f"\n Score Final da Recomendação: {score:.2f}")

mostrar_recomendacoes_user(
    user_id=20,
    dataset=imdb_merged_filtrado,
    ratings_df=ratings_df,
    test_data=test_data,
    train_movie_matrix=train_movie_matrix,
    popular_movies=popular_movies
)


--- Recomendação para o Utilizador 20 ---
Filmes Recomendados:
 - Canary (tt8595480)
 - Fikkefuchs (tt5275476)
 - La peur (tt4741354)
 - Ray (tt4551318)
 - Ekvtime: Man of God (tt6058226)
 - Gansin (tt4844288)
 - Mientras dure la guerra (tt7818580)
 - The Civil War on Drugs (tt2316000)
 - Chicogrande (tt1603471)
 - RengÃ´ kantai shirei chÃ´kan: Yamamoto Isoroku (tt1932695)

 Score Final da Recomendação: 0.60
